##Building Highly Scalable Agents Data Analysis Agents with Teradata AI Studio



![title](images/indatabase.jpg)

In [ ]:
import pandas as pd 
import teradataml as tdml
import getpass
import os 
import httpx
import re

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

In [ ]:
from teradataml import create_context, execute_sql

# Retrieve and store the JWT access token for the current user session.
# The %gen_token magic is registered automatically by the platform on kernel start.
# By default the token is stored in `user_token`.

%gen_token
print("User JWT Access Token:", user_token)

# Default JWT-based connection using dbc (preconfigured Teradata Connection)
eng = create_context(host = 'dbc', logmech = 'JWT', logdata = f'token={user_token}')
result = execute_sql(f"select * from dbc.dbcinfo;")
for row in result:
    print(row)

### DataFrame Initialization (Teradata ML)
Loads a dataset from the Teradata database schema `DEMO_DataScienceExploration`, specifically the `House_Prices` table.
Creates a `tdml.DataFrame` object that enables distributed data processing and machine learning operations directly in-database.
This dataframe is used as the input dataset for downstream analytics such as clustering and feature engineering.

In [4]:
df = tdml.DataFrame(tdml.in_schema('DEMO_DataScienceExploration', 'House_Prices'))

In [13]:
df

id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
54997,2014/12/30,1255780.0,5,4.0,4180.0,12042.0,2.0,0,0,3,10,4180.0,0.0,2014,0,98075,47.5959,-122.014,1800.0,6052.0
4894,2015/03/09,620000.0,5,1.0,2230.0,16800.0,1.5,0,3,4,7,1700.0,530.0,1923,0,98136,47.5161,-122.395,2730.0,18400.0
34136,2014/07/18,590000.0,4,2.5,2290.0,11072.0,2.0,0,0,3,9,2290.0,0.0,1986,0,98074,47.6283,-122.03,2340.0,9774.0
3283,2015/01/28,225000.0,3,1.0,1000.0,9295.0,1.0,0,0,3,7,1000.0,0.0,1955,0,98146,47.483999999999995,-122.346,1320.0,13500.0
77530,2015/01/28,210000.0,3,1.0,1030.0,4583.0,1.0,0,0,3,7,1030.0,0.0,1967,0,98198,47.4231,-122.329,1730.0,8023.0
44128,2014/07/16,242025.0,4,1.75,1400.0,54014.0,1.5,0,0,4,7,1400.0,0.0,1935,0,98031,47.4153,-122.184,1910.0,8523.0
40641,2014/07/17,425000.0,4,1.0,1800.0,12485.0,1.0,0,0,5,7,950.0,850.0,1955,0,98006,47.5729,-122.147,1290.0,9840.0
66600,2015/04/02,577500.0,2,2.5,2330.0,3000.0,2.0,0,3,3,8,2330.0,0.0,1915,1994,98144,47.5953,-122.294,1760.0,4000.0
34809,2014/08/13,510500.0,3,1.0,1270.0,8000.0,1.0,0,0,4,7,1270.0,0.0,1957,0,98007,47.5874,-122.13600000000001,1470.0,8000.0
59626,2014/08/26,434000.0,4,3.0,2010.0,8171.0,2.0,0,0,3,8,2010.0,0.0,1973,0,98011,47.7688,-122.21799999999999,2090.0,8203.0


In [12]:
df.columns

['id',
 'date',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

### Clustering Tool
Performs K-Means clustering on housing data using key property features such as price, size, bedrooms, bathrooms, and floors.
Assigns each property to one of three clusters and computes the average feature values for each cluster.
Returns the cluster-level summary as a JSON object for downstream analysis or AI agent consumption.

In [17]:
@tool
def clustering() -> str:
    '''Get clustering'''

    features = ['price', 'bedrooms','bathrooms','sqft_living','sqft_lot','floors','sqft_above','sqft_basement']
    #KMeans Clustering
    kmeans_model = tdml.KMeans(
        data=df,
        id_column='id',
        target_columns=features,
        num_clusters=3,
        initial_centroids_method='kmeans++',
        seed=10,
        stop_threshold=0.0395,
        max_iter_num=10,
        output_cluster_assignment=True
    )

    
    #Get Results - Averegae by cluster
    result = kmeans_model.result
    features_kmeans = features+['td_clusterid_kmeans']
    df1 = df.join(result, how='inner', on=['id'], lsuffix='t1', rsuffix='t2')[features_kmeans]

    df_local = df1.groupby('td_clusterid_kmeans').mean().to_pandas()
    json_string = df_local.to_json()
    return json_string

tools = [clustering]

In [41]:
api_key = getpass.getpass('Enter your OpenAI API Key: ')

Enter your OpenAI API Key:  ········


### Language Model Configuration
Initializes the Large Language Model (LLM) with the specified model, API credentials, and endpoint for inference.
Configures generation parameters such as maximum response length and request timeout while creating a secure HTTP client.
Provides the core reasoning engine used by the agent to understand requests, invoke tools, and generate responses.

In [43]:
base_url    = os.environ.get('TD_LITELLM_BASE_URL')
model  = os.environ.setdefault("MODEL_NAME","us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile")
#api_key  = input("Enter API key: ").strip()


# Configure the LLM
llm = ChatOpenAI(
    model=model,
    api_key=api_key,
    base_url=base_url, 
    max_tokens=500,
    timeout=30,
    http_client=httpx.Client(verify=False),
)

### ReAct Agent
Creates a ReAct (Reasoning + Acting) agent that follows the provided system prompt and uses only the supplied tools to answer user queries.
The agent reasons about the request, decides when to invoke the clustering tool, and incorporates the results into its response.
Executes the query through the language model while maintaining a tool-constrained workflow.

In [44]:
QUERY = "Give me clustering"
PROMPT = "You are a data scientist agent. Use given tools only"
agent = create_react_agent(llm, tools, prompt=PROMPT)

### Agent Invocation
Executes the ReAct agent by sending a user query wrapped in a `HumanMessage` as part of the message history.
Triggers the agent’s reasoning loop, allowing it to decide whether to call available tools (e.g., clustering) or respond directly.
Returns the final structured response from the agent after tool use and LLM reasoning are completed.

In [45]:
print('Running  Data Agent...\n')
agent_result = agent.invoke({'messages': [HumanMessage(content=QUERY)]})

print('=' * 65)
print('AGENT RESPONSE')
print('=' * 65)
print(agent_result['messages'][-1].content)

Running  Data Agent...

AGENT RESPONSE
Based on the clustering analysis, here are the results showing **3 distinct clusters** of properties:

## Cluster Summary

**Cluster 0 - Budget/Starter Homes:**
- Mean Price: $371,008
- Bedrooms: 3.2
- Bathrooms: 1.9
- Living Space: 1,727 sq ft
- Lot Size: 12,567 sq ft
- Floors: 1.4

**Cluster 1 - Mid-Range Homes:**
- Mean Price: $794,751
- Bedrooms: 3.7
- Bathrooms: 2.5
- Living Space: 2,699 sq ft
- Lot Size: 19,399 sq ft
- Floors: 1.7

**Cluster 2 - Luxury/Premium Homes:**
- Mean Price: $1,978,702
- Bedrooms: 4.2
- Bathrooms: 3.5
- Living Space: 4,320 sq ft
- Lot Size: 25,674 sq ft
- Floors: 1.9

The clustering clearly segments the housing market into three tiers based on price, size, and amenities. Cluster 2 represents luxury properties with significantly larger living spaces and more bedrooms/bathrooms, while Cluster 0 represents more affordable, smaller starter homes.
